# BERT 最小直觉（练习稿，不上博客）

深度学习层，接在 seq2seq Transformer 之后：

- **Word2Vec**：一词一向量（静态）。
- **BERT**：双向 Transformer Encoder + MLM 预训练 → **同词异境异向量**（语境相关），再微调下游任务。

本笔记用极简示意（不下载大模型也可跑通核心对比）；若环境有 `transformers`，可选单元格调用真实模型。


In [ ]:
# 不依赖 GPU：用哈希伪向量演示「静态 vs 语境」
import numpy as np

def static_embed(word, dim=8, seed=0):
    """静态：词形 → 固定向量（模拟 Word2Vec）。"""
    rng = np.random.RandomState(abs(hash((word, seed))) % (2**32))
    v = rng.randn(dim)
    return v / (np.linalg.norm(v) + 1e-8)

def contextual_embed(sentence_tokens, index, dim=8):
    """语境：向量 = 中心词静态向量 + 上下文词静态向量均值（玩具版双向混合）。"""
    center = static_embed(sentence_tokens[index])
    ctx = [static_embed(t) for i, t in enumerate(sentence_tokens) if i != index]
    if not ctx:
        return center
    return 0.6 * center + 0.4 * np.mean(ctx, axis=0)

s1 = ["我", "喜欢", "银行"]
s2 = ["河", "边", "银行"]
# 「银行」在两句中词形相同
i1, i2 = s1.index("银行"), s2.index("银行")
v_static_1 = static_embed("银行")
v_static_2 = static_embed("银行")
v_ctx_1 = contextual_embed(s1, i1)
v_ctx_2 = contextual_embed(s2, i2)

print("静态「银行」两句是否相同:", np.allclose(v_static_1, v_static_2))
print("语境「银行」余弦相似度:", float(v_ctx_1 @ v_ctx_2 / (np.linalg.norm(v_ctx_1) * np.linalg.norm(v_ctx_2))))
print("→ 静态相同；玩具语境向量因上下文不同而不同（真实 BERT 由自注意力实现）。")


## MLM 直觉（掩码语言模型）

预训练时随机遮盖输入中的部分 token，用双向上下文预测被遮盖词。  
因此 Encoder 必须同时看左右上下文——与从左到右的生成式 LM、以及静态 Word2Vec 都不同。

微调：在 `[CLS]`（或池化）向量上接分类头，做句对/单句分类等。


In [ ]:
# 可选：若已安装 transformers + 能下载模型，取消注释运行
# 默认跳过，避免无网环境失败
RUN_HF = False

if RUN_HF:
    from transformers import AutoTokenizer, AutoModel
    import torch
    name = "bert-base-chinese"
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModel.from_pretrained(name)
    model.eval()
    texts = ["我喜欢银行", "河边有一家银行"]
    batch = tok(texts, padding=True, return_tensors="pt")
    with torch.no_grad():
        out = model(**batch).last_hidden_state
    # 取「银」字位置做极粗对比（真实应对齐 subword）
    print(out.shape)
else:
    print("RUN_HF=False：跳过 HuggingFace 调用。玩具语境对比见上一格。")


## 与目录中其他篇的关系

- `06` / `seq2seq-transformer`：任务上训 Encoder–Decoder（翻译）。  
- 本篇：Encoder-only + 预训练目标（MLM）→ 表示 / 微调。  
- 博客「大模型」层：把生成式 LLM 当 API；BERT 仍归深度学习「学表示」。
